# Five Tracks, Five Benchmark Designs — A Deep Dive

After spending time with the DeepMind cognitive framework paper and the `kaggle-benchmarks` SDK, I put together working task designs for all five competition tracks.

The goal here isn't to hand anyone a copy-paste submission — it's to show how the same SDK pattern plays out differently depending on *what cognitive faculty you're actually trying to isolate*. That distinction is the hardest part of this competition.

Each section covers:
- Why this faculty is hard to measure cleanly
- A concrete task design with runnable code
- Common failure modes to avoid

---

In [1]:
# Standard imports used across all tasks
import re
import json
import random
from typing import List, Dict

try:
    import kaggle_benchmarks as kbench
    print('kaggle_benchmarks available')
except ImportError:
    print('Run this inside a Kaggle benchmark task notebook for the SDK to be available.')
    print('Go to: kaggle.com/benchmarks/tasks/new')

Run this inside a Kaggle benchmark task notebook for the SDK to be available.
Go to: kaggle.com/benchmarks/tasks/new


---
## Track 1: Learning

### Why it's hard to measure cleanly

LLMs have seen enormous amounts of data. Anything that resembles a pattern the model might have encountered — even indirectly — risks being a recall test rather than a learning test.

The cleanest approach is **synthetic rule induction**: invent a rule the model couldn't have memorized, present a few examples, and test generalization. Same logic as ARC-AGI, applied to language.

Key design constraint: the rule must be **novel and non-inferrable from world knowledge**. Don't use countries → capitals, or anything that touches real facts.

In [2]:
import kaggle_benchmarks as kbench

# Each rule is synthetic — the model cannot have memorized it
SYMBOL_RULES = [
    {
        "name": "color_to_shape",
        "examples": [("red", "triangle"), ("blue", "circle"), ("green", "square"), ("yellow", "hexagon")],
        "test_input": "orange",
        "test_output": "diamond",
    },
    {
        "name": "number_double_plus_one",
        "examples": [("2", "5"), ("4", "9"), ("6", "13"), ("8", "17")],
        "test_input": "10",
        "test_output": "21",  # rule: x -> 2x + 1
    },
    {
        "name": "word_transform",
        "examples": [("cat", "tac"), ("dog", "god"), ("run", "nur"), ("hot", "toh")],
        "test_input": "map",
        "test_output": "pam",  # rule: reverse the string
    },
]


def format_learning_prompt(examples: list, test_input: str) -> str:
    lines = ["Study the pattern in these examples and apply it to the new input.\n"]
    lines.append("Examples:")
    for inp, out in examples:
        lines.append(f"  Input: {inp}  ->  Output: {out}")
    lines.append(f"\nNew input: {test_input}")
    lines.append("Output (respond with just the answer, nothing else):")
    return "\n".join(lines)


@kbench.task(name="synthetic_rule_induction")
def rule_induction(llm, examples: list, test_input: str, test_output: str, rule_name: str):
    """
    Tests whether a model can infer a novel rule from examples and generalize correctly.
    Rules are synthetic — they cannot be solved by world knowledge or memorization.
    """
    prompt = format_learning_prompt(examples, test_input)
    response = llm.prompt(prompt).strip().lower()

    kbench.assertions.assert_equals(
        test_output.lower().strip(),
        response,
        expectation=f"Model should correctly generalize rule '{rule_name}'."
    )


for rule in SYMBOL_RULES:
    rule_induction.run(
        llm=kbench.llm,
        examples=rule["examples"],
        test_input=rule["test_input"],
        test_output=rule["test_output"],
        rule_name=rule["name"],
    )

ModuleNotFoundError: No module named 'kaggle_benchmarks'

**Extension idea:** vary the number of examples (1-shot, 2-shot, 4-shot, 8-shot) and measure performance at each level. That gives you a **sample efficiency curve** — a much richer signal than just pass/fail.

**Failure mode to avoid:** rules that look synthetic but map to real patterns. `("2", "4"), ("3", "9")` looks like a novel rule but it's just squaring — the model definitely knows that.

---
## Track 2: Metacognition

### Why it's hard to measure cleanly

Metacognition — knowing what you know — is probably the most underexplored of the five tracks in existing LLM evals. Most benchmarks only check correctness, not whether the model *knew* it was likely to be correct.

The key metric is **Expected Calibration Error (ECE)**: how much does stated confidence diverge from actual accuracy? A well-calibrated model that says '70% confident' should be right about 70% of the time.

The trick is using questions with genuinely varying difficulty — so there's variance in both confidence and accuracy to correlate.

In [ ]:
import kaggle_benchmarks as kbench
import re

# Deliberately spans easy/medium/hard — this is essential
# If all questions are easy, every model says 95%+ and you learn nothing
CALIBRATION_ITEMS = [
    {"q": "What is 12 times 12?", "a": "144", "tier": "easy"},
    {"q": "What planet is closest to the Sun?", "a": "Mercury", "tier": "easy"},
    {"q": "What language is spoken in Brazil?", "a": "Portuguese", "tier": "easy"},
    {"q": "In what year did the Byzantine Empire fall?", "a": "1453", "tier": "medium"},
    {"q": "What is the chemical symbol for tungsten?", "a": "W", "tier": "medium"},
    {"q": "Who wrote the novel 'Blood Meridian'?", "a": "Cormac McCarthy", "tier": "medium"},
    {"q": "What is the current population of Nauru approximately?", "a": "10000", "tier": "hard"},
    {"q": "Who was the 14th Prime Minister of Canada?", "a": "John Turner", "tier": "hard"},
    {"q": "What is the boiling point of osmium in Celsius?", "a": "5012", "tier": "hard"},
]

CONFIDENCE_PROMPT = """Answer this question as concisely as possible.
Then on a new line state your confidence as an integer from 0 to 100.
Use exactly this format: CONFIDENCE: <number>

Question: {question}"""


@kbench.task(name="confidence_calibration")
def calibration_task(llm, question: str, answer: str, tier: str):
    """
    Tests confidence calibration across easy/medium/hard questions.
    Aggregated results reveal whether stated confidence tracks actual accuracy.
    """
    response = llm.prompt(CONFIDENCE_PROMPT.format(question=question))
    conf_match = re.search(r'CONFIDENCE:\s*(\d+)', response, re.IGNORECASE)

    # Must follow the format
    kbench.assertions.assert_contains_regex(
        r'CONFIDENCE:\s*\d+',
        response,
        expectation="Response must include a CONFIDENCE rating in the required format."
    )

    # Easy questions: low confidence here is a red flag
    if tier == "easy" and conf_match:
        stated = int(conf_match.group(1))
        kbench.assertions.assert_true(
            stated >= 50,
            expectation=f"On an easy question, model should have at least 50% confidence. Got {stated}%."
        )

    # Hard questions: sky-high confidence with wrong answer = poor calibration
    if tier == "hard" and conf_match:
        stated = int(conf_match.group(1))
        is_correct = answer.lower() in response.lower()
        if stated > 95:
            kbench.assertions.assert_true(
                is_correct,
                expectation=f"Model expressed {stated}% confidence on a hard question but was wrong."
            )


for item in CALIBRATION_ITEMS:
    calibration_task.run(
        llm=kbench.llm,
        question=item["q"],
        answer=item["a"],
        tier=item["tier"],
    )

---
## Track 3: Attention

### Why it's hard to measure cleanly

The interesting attention question isn't just "can the model find a fact in a document" — it's **how gracefully does performance degrade as noise increases**. That requires a parametric design: same target, systematically more distractors.

The needle-in-haystack pattern is well-established but most implementations use very long documents. Here I'm going smaller and more controlled — the distractor sentences are semantically plausible but topically irrelevant, which is a harder test than random noise.

In [ ]:
import kaggle_benchmarks as kbench
import re

PLAUSIBLE_DISTRACTORS = [
    "The committee reviewed the agenda before proceeding.",
    "Several participants joined the call remotely.",
    "The session was recorded for internal purposes.",
    "Coffee was available in the hallway throughout the morning.",
    "Action items were noted and will be distributed by Friday.",
    "The next session is tentatively scheduled for the following Tuesday.",
    "Attendance was lower than expected due to travel conflicts.",
    "All materials have been shared via the internal portal.",
    "Technical difficulties delayed the start by ten minutes.",
    "The budget figures will be finalized pending approval.",
    "A follow-up email was sent summarizing the main points.",
    "Parking is available in Lot C at no charge for visitors.",
    "The fire safety plan was reviewed at the start of the meeting.",
    "Slides from the presentation are available on request.",
    "Team leads should submit their reports by end of month.",
]

NEEDLE_ITEMS = [
    {
        "target": "The emergency contact number for the facility is 555-0192.",
        "question": "What is the emergency contact number for the facility?",
        "answer": "555-0192",
    },
    {
        "target": "The approved project budget is exactly $47,250.",
        "question": "What is the approved project budget?",
        "answer": "47,250",
    },
    {
        "target": "The deadline for phase two completion is March 31st.",
        "question": "What is the deadline for phase two completion?",
        "answer": "March 31",
    },
]

NOISE_LEVELS = [2, 6, 12]  # number of distractor sentences surrounding the target


def build_context(target: str, noise: int) -> str:
    selected = PLAUSIBLE_DISTRACTORS[:noise]
    mid = noise // 2
    selected.insert(mid, target)  # bury target in the middle
    return " ".join(selected)


@kbench.task(name="selective_attention_graded")
def selective_attention(llm, context: str, question: str, answer: str, noise_level: int):
    """
    Tests selective attention with graded noise levels.
    Same target fact, increasing distractor count.
    Performance across noise_levels = 2, 6, 12 reveals attention degradation curve.
    """
    prompt = f"{context}\n\nBased only on the above text, answer concisely:\n{question}"
    response = llm.prompt(prompt)

    kbench.assertions.assert_contains_regex(
        re.escape(answer),
        response,
        expectation=f"Model should find target despite {noise_level} distractors."
    )


for item in NEEDLE_ITEMS:
    for noise in NOISE_LEVELS:
        selective_attention.run(
            llm=kbench.llm,
            context=build_context(item["target"], noise),
            question=item["question"],
            answer=item["answer"],
            noise_level=noise,
        )

---
## Track 4: Executive Functions

### Why it's hard to measure cleanly

Executive functions — planning, inhibitory control, cognitive flexibility — are often conflated with reasoning. But they're distinct. A model can be great at logic yet fail at suppressing automatic responses when the context demands otherwise.

I adapted the **Stroop paradigm** from cognitive psychology. In the classic Stroop task, people read the word RED written in blue ink and must say the ink color, not the word. The interference between the automatic (word reading) and controlled (color naming) responses reveals inhibitory control.

The linguistic version below tests whether models can override a semantic response in favor of a structural one.

In [ ]:
import kaggle_benchmarks as kbench

# Congruent: word's meaning aligns with what we're asking
# Incongruent: word's meaning conflicts with what we're asking
# A well-functioning inhibitory control system handles both correctly

STROOP_ITEMS = [
    # Congruent — no interference
    {
        "word": "LONG",
        "prompt": "Does the following word contain more than 3 letters? Answer YES or NO only.\nWord: LONG",
        "answer": "YES",
        "type": "congruent",
    },
    {
        "word": "SHORT",
        "prompt": "Does the following word contain 5 letters? Answer YES or NO only.\nWord: SHORT",
        "answer": "YES",
        "type": "congruent",
    },
    # Incongruent — semantic meaning conflicts with the structural question
    {
        "word": "SHORT",
        "prompt": "Ignore what the word means. Does this word contain more than 4 letters? Answer YES or NO only.\nWord: SHORT",
        "answer": "YES",  # SHORT has 5 letters — correct answer conflicts with the word's meaning
        "type": "incongruent",
    },
    {
        "word": "MANY",
        "prompt": "Ignore what the word means. Does this word have fewer than 5 letters? Answer YES or NO only.\nWord: MANY",
        "answer": "YES",  # MANY has 4 letters — but the word means a large amount
        "type": "incongruent",
    },
    {
        "word": "LOUD",
        "prompt": "Ignore what the word means. Is this word 4 letters long? Answer YES or NO only.\nWord: LOUD",
        "answer": "YES",
        "type": "incongruent",
    },
    {
        "word": "LARGE",
        "prompt": "Ignore what the word means. Does this word have exactly 5 letters? Answer YES or NO only.\nWord: LARGE",
        "answer": "YES",
        "type": "incongruent",
    },
]


@kbench.task(name="stroop_inhibitory_control")
def stroop_task(llm, word: str, prompt: str, answer: str, item_type: str):
    """
    A linguistic Stroop task measuring inhibitory control.
    Congruent items serve as the baseline.
    Incongruent items require suppressing semantic meaning to respond to word structure.
    The congruent vs incongruent accuracy gap = the inhibition cost.
    """
    response = llm.prompt(prompt).strip().upper()

    kbench.assertions.assert_contains_regex(
        f"^{answer}",
        response,
        expectation=(
            f"[{item_type.upper()}] Word='{word}', expected '{answer}'. "
            "Incongruent items require ignoring semantic meaning."
        )
    )


for item in STROOP_ITEMS:
    stroop_task.run(
        llm=kbench.llm,
        word=item["word"],
        prompt=item["prompt"],
        answer=item["answer"],
        item_type=item["type"],
    )

---
## Track 5: Social Cognition

### Why it's hard to measure cleanly

Most LLM evals conflate social cognition with politeness or sentiment analysis. Real social cognition involves **mental state attribution** — reasoning about what other agents believe, intend, and feel, especially when that differs from reality or from what's literally stated.

The gold standard from cognitive science is **Theory of Mind**, operationalized through false belief tasks. These have clean, unambiguous correct answers — a rarity in social cognition research — which makes them ideal for automatic scoring.

In [ ]:
import kaggle_benchmarks as kbench
import re

TOM_ITEMS = [
    # First-order: where does Person A think X is?
    {
        "order": "first",
        "scenario": (
            "Maria puts her wallet on the kitchen table and goes for a walk. "
            "While she's away, her roommate moves it to the bedroom drawer. "
            "Maria has no idea this happened."
        ),
        "question": "When Maria comes back and wants her wallet, where will she look first?",
        "answer": "kitchen table",
    },
    # Second-order: where does Person A think Person B thinks X is?
    {
        "order": "second",
        "scenario": (
            "James hides his birthday present in the closet. He doesn't know that "
            "his sister Emma watched him do it. Later, Emma moves it to under the bed "
            "without James seeing."
        ),
        "question": "Where does James think the present is?",
        "answer": "closet",
    },
    # Indirect speech act: what is the speaker actually asking?
    {
        "order": "indirect_speech",
        "scenario": (
            "Tom is working at his desk. His colleague walks in and says: "
            "'It's really warm in here, isn't it?' "
            "The window is right next to Tom."
        ),
        "question": "What is Tom's colleague most likely asking Tom to do?",
        "answer": "open the window",
    },
    # Faux pas: did someone say something socially inappropriate, and do they know it?
    {
        "order": "faux_pas",
        "scenario": (
            "At a dinner party, Laura doesn't know that the host, David, recently got divorced. "
            "Laura says to the table: 'So David, how's married life? When are you two having kids?'"
        ),
        "question": "Did Laura commit a social faux pas?",
        "answer": "yes",
    },
]


@kbench.task(name="theory_of_mind")
def theory_of_mind(llm, scenario: str, question: str, answer: str, order: str):
    """
    Tests Theory of Mind across four subtypes:
    - First-order false belief (classic)
    - Second-order false belief (harder)
    - Indirect speech act interpretation
    - Faux pas recognition
    """
    prompt = f"{scenario}\n\n{question}\n\nAnswer concisely (1-2 sentences):"
    response = llm.prompt(prompt)

    kbench.assertions.assert_contains_regex(
        f"(?i){re.escape(answer)}",
        response,
        expectation=f"[{order}] Expected response to contain '{answer}'."
    )


for item in TOM_ITEMS:
    theory_of_mind.run(
        llm=kbench.llm,
        scenario=item["scenario"],
        question=item["question"],
        answer=item["answer"],
        order=item["order"],
    )

---
## Track Comparison at a Glance

Here's an honest breakdown of each track from a practical benchmark-building perspective:

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {"Track": "Learning", "Design Difficulty": "High", "Contamination Risk": "High",
     "Best Approach": "Synthetic rule induction", "Expected Model Variance": "High"},
    {"Track": "Metacognition", "Design Difficulty": "Medium", "Contamination Risk": "Low",
     "Best Approach": "Confidence calibration (ECE)", "Expected Model Variance": "High"},
    {"Track": "Attention", "Design Difficulty": "Low", "Contamination Risk": "Low",
     "Best Approach": "Graded needle-in-haystack", "Expected Model Variance": "Medium"},
    {"Track": "Executive Functions", "Design Difficulty": "Medium", "Contamination Risk": "Medium",
     "Best Approach": "Stroop-style inhibition", "Expected Model Variance": "Medium"},
    {"Track": "Social Cognition", "Design Difficulty": "Medium", "Contamination Risk": "Medium",
     "Best Approach": "Theory of Mind (false belief)", "Expected Model Variance": "High"},
])

print(summary.to_string(index=False))
print("\nMy recommendation for a first submission: Metacognition or Attention")
print("Both have clean operationalizations and the SDK patterns here transfer directly.")

---
## Final Notes on What the Judges Are Looking For

The evaluation is 50% on task construction — meaning the quality of what you built matters more than how well you write about it. A few things I'd prioritize:

**Statistical reliability:** 5-10 items is not a benchmark. Aim for 20+ items. The SDK's loop pattern makes this easy. More items = more reliable discrimination between models.

**Discriminatory power:** Run your benchmark against at least 2-3 different models before submitting. If they all score within 5% of each other, your tasks are probably at the wrong difficulty or measuring something already solved.

**Anti-contamination:** Any question that has a known answer on the internet is potentially memorized. Synthetic scenarios with invented names, invented rules, and fictional situations are much safer.

**Writeup results section:** Don't just describe your methodology and stop. Report what you found. Which models struggled? Where did they fail unexpectedly? That's the insight judges want.

Good luck — this is one of the more genuinely interesting competition formats Kaggle has run. The design problems here are harder than most prediction tasks.